In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/website-classification/website_classification.csv


In [2]:
import pandas as pd

# Replace with your actual file path
df = pd.read_csv('/kaggle/input/website-classification/website_classification.csv')

# Display the first few rows
print(df.head())

   Unnamed: 0                                        website_url  \
0           0     https://www.booking.com/index.html?aid=1743217   
1           1                   https://travelsites.com/expedia/   
2           2               https://travelsites.com/tripadvisor/   
3           3              https://www.momondo.in/?ispredir=true   
4           4  https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...   

                                cleaned_website_text Category  
0  official site good hotel accommodation big sav...   Travel  
1  expedia hotel book sites like use vacation wor...   Travel  
2  tripadvisor hotel book sites like previously d...   Travel  
3  cheap flights search compare flights momondo f...   Travel  
4  bot create free account create free account si...   Travel  


In [3]:
df

,Unnamed: 0,website_url,cleaned_website_text,Category
0,0,https://www.booking.com/index.html?aid=1743217,official site good hotel accommodation big sav...,Travel
1,1,https://travelsites.com/expedia/,expedia hotel book sites like use vacation wor...,Travel
2,2,https://travelsites.com/tripadvisor/,tripadvisor hotel book sites like previously d...,Travel
3,3,https://www.momondo.in/?ispredir=true,cheap flights search compare flights momondo f...,Travel
4,4,https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...,bot create free account create free account si...,Travel
...,...,...,...,...
1403,1403,http://www.oldwomen.org/,old nude women porn mature granny sex horny ol...,Adult
1404,1404,http://www.webcamslave.com,bdsm cams bdsm chat bondage cams free bdsm vid...,Adult
1405,1405,http://www.buyeuroporn.com/,porno dvd online european porn dvd cheap adult...,Adult
1406,1406,http://www.analdreamhouse.com/30/03/agecheck/i...,anal dream house anal dream house anal dream h...,Adult


In [4]:
df['Category'].unique()

array(['Travel', 'Social Networking and Messaging', 'News',
       'Streaming Services', 'Sports', 'Photography',
       'Law and Government', 'Health and Fitness', 'Games', 'E-Commerce',
       'Forums', 'Food', 'Education', 'Computers and Technology',
       'Business/Corporate', 'Adult'], dtype=object)

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Category_b'] = le.fit_transform(df['Category'])

In [6]:
pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 5.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
import contractions
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()  # enables progress_apply for a progress bar

# Apply contraction expansion to each row
df['expanded_text'] = df['cleaned_website_text'].progress_apply(lambda x: contractions.fix(str(x)))

  0%|          | 0/1408 [00:00<?, ?it/s]

In [8]:
df

,Unnamed: 0,website_url,cleaned_website_text,Category,Category_b,expanded_text
0,0,https://www.booking.com/index.html?aid=1743217,official site good hotel accommodation big sav...,Travel,15,official site good hotel accommodation big sav...
1,1,https://travelsites.com/expedia/,expedia hotel book sites like use vacation wor...,Travel,15,expedia hotel book sites like use vacation wor...
2,2,https://travelsites.com/tripadvisor/,tripadvisor hotel book sites like previously d...,Travel,15,tripadvisor hotel book sites like previously d...
3,3,https://www.momondo.in/?ispredir=true,cheap flights search compare flights momondo f...,Travel,15,cheap flights search compare flights momondo f...
4,4,https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...,bot create free account create free account si...,Travel,15,bot create free account create free account si...
...,...,...,...,...,...,...
1403,1403,http://www.oldwomen.org/,old nude women porn mature granny sex horny ol...,Adult,0,old nude women porn mature granny sex horny ol...
1404,1404,http://www.webcamslave.com,bdsm cams bdsm chat bondage cams free bdsm vid...,Adult,0,bdsm cams bdsm chat bondage cams free bdsm vid...
1405,1405,http://www.buyeuroporn.com/,porno dvd online european porn dvd cheap adult...,Adult,0,porno dvd online european porn dvd cheap adult...
1406,1406,http://www.analdreamhouse.com/30/03/agecheck/i...,anal dream house anal dream house anal dream h...,Adult,0,anal dream house anal dream house anal dream h...


In [9]:
print(df['expanded_text'].head())
print(df['expanded_text'].apply(type).value_counts())

0    official site good hotel accommodation big sav...
1    expedia hotel book sites like use vacation wor...
2    tripadvisor hotel book sites like previously d...
3    cheap flights search compare flights momondo f...
4    bot create free account create free account si...
Name: expanded_text, dtype: object
expanded_text
<class 'str'>    1408
Name: count, dtype: int64


In [10]:
import re
import string
import pandas as pd
from tqdm import tqdm
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer, WordNetLemmatizer

tqdm.pandas()

def clean_text(text, spell_correct=False, remove_stopwords=True):
    if not isinstance(text, str):
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+|https\S+', '', text)

    # 3. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # 3. REMOVE datetime FIRST ✅
    text = re.sub(
        r'on\s+\w+,\s+\d{4}-\d{2}-\d{2}\s+at\s+\d{1,2}:\d{2}',
        '',
        text,
        flags=re.IGNORECASE
    )

    # 4. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # 5. Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)

    # 6. Remove hashtags (#hashtag)
    text = re.sub(r'#\w+', '', text)

    # 7. Remove repeated characters (heeeello -> helo)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 8. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 9. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 10. Remove extra whitespace
    text = ' '.join(text.split())

    # 11. Tokenize
    tokens = text.split()

    # 12. Remove stopwords (optional)
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        tokens = [w for w in tokens if w not in stop_words and len(w) >= 2]
    else:
        # Just filter short words
        tokens = [w for w in tokens if len(w) >= 2]
    """
    # 13. Spell correction (optional and SLOW)
    if spell_correct:
        tokens = [str(TextBlob(w).correct()) for w in tokens]
    """
    # Return cleaned tokens as string
    return ' '.join(tokens)


# Step 1: Clean the text
print("=== Step 1: Cleaning Text ===")
df['cleaned_text'] = df['expanded_text'].progress_apply(
    lambda x: clean_text(x, spell_correct=False, remove_stopwords=True)
)

print(f"Cleaning completed. Sample:\n{df['cleaned_text'].head(3)}")

=== Step 1: Cleaning Text ===


100%|██████████| 1408/1408 [00:01<00:00, 742.13it/s]

Cleaning completed. Sample:
0    official site good hotel accommodation big sav...
1    expedia hotel book sites like use vacation wor...
2    tripadvisor hotel book sites like previously d...
Name: cleaned_text, dtype: object


In [11]:
df

,Unnamed: 0,website_url,cleaned_website_text,Category,Category_b,expanded_text,cleaned_text
0,0,https://www.booking.com/index.html?aid=1743217,official site good hotel accommodation big sav...,Travel,15,official site good hotel accommodation big sav...,official site good hotel accommodation big sav...
1,1,https://travelsites.com/expedia/,expedia hotel book sites like use vacation wor...,Travel,15,expedia hotel book sites like use vacation wor...,expedia hotel book sites like use vacation wor...
2,2,https://travelsites.com/tripadvisor/,tripadvisor hotel book sites like previously d...,Travel,15,tripadvisor hotel book sites like previously d...,tripadvisor hotel book sites like previously d...
3,3,https://www.momondo.in/?ispredir=true,cheap flights search compare flights momondo f...,Travel,15,cheap flights search compare flights momondo f...,cheap flights search compare flights momondo f...
4,4,https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...,bot create free account create free account si...,Travel,15,bot create free account create free account si...,bot create free account create free account si...
...,...,...,...,...,...,...,...
1403,1403,http://www.oldwomen.org/,old nude women porn mature granny sex horny ol...,Adult,0,old nude women porn mature granny sex horny ol...,old nude women porn mature granny sex horny ol...
1404,1404,http://www.webcamslave.com,bdsm cams bdsm chat bondage cams free bdsm vid...,Adult,0,bdsm cams bdsm chat bondage cams free bdsm vid...,bdsm cams bdsm chat bondage cams free bdsm vid...
1405,1405,http://www.buyeuroporn.com/,porno dvd online european porn dvd cheap adult...,Adult,0,porno dvd online european porn dvd cheap adult...,porno dvd online european porn dvd cheap adult...
1406,1406,http://www.analdreamhouse.com/30/03/agecheck/i...,anal dream house anal dream house anal dream h...,Adult,0,anal dream house anal dream house anal dream h...,anal dream house anal dream house anal dream h...


In [12]:
# Using NLTK
import nltk
nltk.download('punkt')

def tokenize_text(text):
    # Word tokenization
    word_tokens = nltk.word_tokenize(text)
    # Sentence tokenization
    sentence_tokens = nltk.sent_tokenize(text)
    return word_tokens, sentence_tokens

#df_msg['word_tokens'] = tokenize_text(df_msg['cleaned_text'])
df[['word_tokens', 'sentence_tokens']] = df['cleaned_text'].progress_apply(
    lambda x: pd.Series(tokenize_text(x))
)

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
100%|██████████| 1408/1408 [00:04<00:00, 333.70it/s]


In [13]:
!pip install pyspellchecker

from tqdm.notebook import tqdm
import pandas as pd
from spellchecker import SpellChecker

def correct_spelling(tokens):
    spell = SpellChecker()
    corrected = [spell.correction(token) for token in tokens]
    return corrected

#df_msg['word_tokens'] = tokenize_text(df_msg['cleaned_text'])
df['correct_tokens'] = df['sentence_tokens'].progress_apply(
    lambda x: pd.Series(correct_spelling(x))
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 69.6 MB/s eta 0:00:00:00:010:01


100%|██████████| 1408/1408 [04:56<00:00,  4.75it/s]


In [14]:
df

,Unnamed: 0,website_url,cleaned_website_text,Category,Category_b,expanded_text,cleaned_text,word_tokens,sentence_tokens,correct_tokens
0,0,https://www.booking.com/index.html?aid=1743217,official site good hotel accommodation big sav...,Travel,15,official site good hotel accommodation big sav...,official site good hotel accommodation big sav...,"[official, site, good, hotel, accommodation, b...",[official site good hotel accommodation big sa...,official site good hotel accommodation big sav...
1,1,https://travelsites.com/expedia/,expedia hotel book sites like use vacation wor...,Travel,15,expedia hotel book sites like use vacation wor...,expedia hotel book sites like use vacation wor...,"[expedia, hotel, book, sites, like, use, vacat...",[expedia hotel book sites like use vacation wo...,expedia hotel book sites like use vacation wor...
2,2,https://travelsites.com/tripadvisor/,tripadvisor hotel book sites like previously d...,Travel,15,tripadvisor hotel book sites like previously d...,tripadvisor hotel book sites like previously d...,"[tripadvisor, hotel, book, sites, like, previo...",[tripadvisor hotel book sites like previously ...,tripadvisor hotel book sites like previously d...
3,3,https://www.momondo.in/?ispredir=true,cheap flights search compare flights momondo f...,Travel,15,cheap flights search compare flights momondo f...,cheap flights search compare flights momondo f...,"[cheap, flights, search, compare, flights, mom...",[cheap flights search compare flights momondo ...,cheap flights search compare flights momondo f...
4,4,https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...,bot create free account create free account si...,Travel,15,bot create free account create free account si...,bot create free account create free account si...,"[bot, create, free, account, create, free, acc...",[bot create free account create free account s...,bot create free account create free account si...
...,...,...,...,...,...,...,...,...,...,...
1403,1403,http://www.oldwomen.org/,old nude women porn mature granny sex horny ol...,Adult,0,old nude women porn mature granny sex horny ol...,old nude women porn mature granny sex horny ol...,"[old, nude, women, porn, mature, granny, sex, ...",[old nude women porn mature granny sex horny o...,old nude women porn mature granny sex horny ol...
1404,1404,http://www.webcamslave.com,bdsm cams bdsm chat bondage cams free bdsm vid...,Adult,0,bdsm cams bdsm chat bondage cams free bdsm vid...,bdsm cams bdsm chat bondage cams free bdsm vid...,"[bdsm, cams, bdsm, chat, bondage, cams, free, ...",[bdsm cams bdsm chat bondage cams free bdsm vi...,bdsm cams bdsm chat bondage cams free bdsm vid...
1405,1405,http://www.buyeuroporn.com/,porno dvd online european porn dvd cheap adult...,Adult,0,porno dvd online european porn dvd cheap adult...,porno dvd online european porn dvd cheap adult...,"[porno, dvd, online, european, porn, dvd, chea...",[porno dvd online european porn dvd cheap adul...,porno dvd online european porn dvd cheap adult...
1406,1406,http://www.analdreamhouse.com/30/03/agecheck/i...,anal dream house anal dream house anal dream h...,Adult,0,anal dream house anal dream house anal dream h...,anal dream house anal dream house anal dream h...,"[anal, dream, house, anal, dream, house, anal,...",[anal dream house anal dream house anal dream ...,anal dream house anal dream house anal dream h...


In [15]:
#save it in a csv file after correct spelling
df.to_csv('website_classification_correct_spelling.csv', index=False)

In [16]:
import os
os.listdir('.')

['.virtual_documents', 'website_classification_correct_spelling.csv']

In [17]:
import pandas as pd

# Replace with your actual file path
df2 = pd.read_csv('website_classification_correct_spelling.csv')

# Display the first few rows
print(df2.head())

   Unnamed: 0                                        website_url  \
0           0     https://www.booking.com/index.html?aid=1743217   
1           1                   https://travelsites.com/expedia/   
2           2               https://travelsites.com/tripadvisor/   
3           3              https://www.momondo.in/?ispredir=true   
4           4  https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...   

                                cleaned_website_text Category  Category_b  \
0  official site good hotel accommodation big sav...   Travel          15   
1  expedia hotel book sites like use vacation wor...   Travel          15   
2  tripadvisor hotel book sites like previously d...   Travel          15   
3  cheap flights search compare flights momondo f...   Travel          15   
4  bot create free account create free account si...   Travel          15   

                                       expanded_text  \
0  official site good hotel accommodation big sav...   
1  expedia hotel

In [18]:
df2

,Unnamed: 0,website_url,cleaned_website_text,Category,Category_b,expanded_text,cleaned_text,word_tokens,sentence_tokens,correct_tokens
0,0,https://www.booking.com/index.html?aid=1743217,official site good hotel accommodation big sav...,Travel,15,official site good hotel accommodation big sav...,official site good hotel accommodation big sav...,"['official', 'site', 'good', 'hotel', 'accommo...",['official site good hotel accommodation big s...,official site good hotel accommodation big sav...
1,1,https://travelsites.com/expedia/,expedia hotel book sites like use vacation wor...,Travel,15,expedia hotel book sites like use vacation wor...,expedia hotel book sites like use vacation wor...,"['expedia', 'hotel', 'book', 'sites', 'like', ...",['expedia hotel book sites like use vacation w...,expedia hotel book sites like use vacation wor...
2,2,https://travelsites.com/tripadvisor/,tripadvisor hotel book sites like previously d...,Travel,15,tripadvisor hotel book sites like previously d...,tripadvisor hotel book sites like previously d...,"['tripadvisor', 'hotel', 'book', 'sites', 'lik...",['tripadvisor hotel book sites like previously...,tripadvisor hotel book sites like previously d...
3,3,https://www.momondo.in/?ispredir=true,cheap flights search compare flights momondo f...,Travel,15,cheap flights search compare flights momondo f...,cheap flights search compare flights momondo f...,"['cheap', 'flights', 'search', 'compare', 'fli...",['cheap flights search compare flights momondo...,cheap flights search compare flights momondo f...
4,4,https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...,bot create free account create free account si...,Travel,15,bot create free account create free account si...,bot create free account create free account si...,"['bot', 'create', 'free', 'account', 'create',...",['bot create free account create free account ...,bot create free account create free account si...
...,...,...,...,...,...,...,...,...,...,...
1403,1403,http://www.oldwomen.org/,old nude women porn mature granny sex horny ol...,Adult,0,old nude women porn mature granny sex horny ol...,old nude women porn mature granny sex horny ol...,"['old', 'nude', 'women', 'porn', 'mature', 'gr...",['old nude women porn mature granny sex horny ...,old nude women porn mature granny sex horny ol...
1404,1404,http://www.webcamslave.com,bdsm cams bdsm chat bondage cams free bdsm vid...,Adult,0,bdsm cams bdsm chat bondage cams free bdsm vid...,bdsm cams bdsm chat bondage cams free bdsm vid...,"['bdsm', 'cams', 'bdsm', 'chat', 'bondage', 'c...",['bdsm cams bdsm chat bondage cams free bdsm v...,bdsm cams bdsm chat bondage cams free bdsm vid...
1405,1405,http://www.buyeuroporn.com/,porno dvd online european porn dvd cheap adult...,Adult,0,porno dvd online european porn dvd cheap adult...,porno dvd online european porn dvd cheap adult...,"['porno', 'dvd', 'online', 'european', 'porn',...",['porno dvd online european porn dvd cheap adu...,porno dvd online european porn dvd cheap adult...
1406,1406,http://www.analdreamhouse.com/30/03/agecheck/i...,anal dream house anal dream house anal dream h...,Adult,0,anal dream house anal dream house anal dream h...,anal dream house anal dream house anal dream h...,"['anal', 'dream', 'house', 'anal', 'dream', 'h...",['anal dream house anal dream house anal dream...,anal dream house anal dream house anal dream h...


In [19]:
# ============================================================================
# FUNCTION 2: STEMMING AND LEMMATIZATION
# ============================================================================

def apply_stemming_lemmatization(text):
    if not isinstance(text, str) or not text.strip():
        return {
            "lemma": "",
            "porter": "",
            "lancaster": "",
            "snowball": ""
        }
    
    # Tokenize
    tokens = text.split()
    
    # Initialize stemmers and lemmatizer
    porter = PorterStemmer()
    lancaster = LancasterStemmer()
    snowball = SnowballStemmer("english")
    lemmatizer = WordNetLemmatizer()
    
    # Apply each technique
    lemma_out = [lemmatizer.lemmatize(w) for w in tokens]
    porter_out = [porter.stem(w) for w in tokens]
    lancaster_out = [lancaster.stem(w) for w in tokens]
    snowball_out = [snowball.stem(w) for w in tokens]
    
    # Return all outputs
    return {
        "lemma": " ".join(lemma_out),
        "porter": " ".join(porter_out),
        "lancaster": " ".join(lancaster_out),
        "snowball": " ".join(snowball_out)
    }

# Step 2: Apply stemming and lemmatization
print("\n=== Step 2: Applying Stemming/Lemmatization ===")
df_stemmed = df2['cleaned_text'].progress_apply(
    apply_stemming_lemmatization
).apply(pd.Series)# Combine with original dataframe


=== Step 2: Applying Stemming/Lemmatization ===


100%|██████████| 1408/1408 [00:48<00:00, 29.12it/s]


In [20]:
df_stemmed

,lemma,porter,lancaster,snowball
0,official site good hotel accommodation big sav...,offici site good hotel accommod big save hotel...,off sit good hotel accommod big sav hotel dest...,offici site good hotel accommod big save hotel...
1,expedia hotel book site like use vacation work...,expedia hotel book site like use vacat work ha...,exped hotel book sit lik us vac work hard year...,expedia hotel book site like use vacat work ha...
2,tripadvisor hotel book site like previously de...,tripadvisor hotel book site like previous deal...,tripadv hotel book sit lik prevy deal predomin...,tripadvisor hotel book site like previous deal...
3,cheap flight search compare flight momondo fin...,cheap flight search compar flight momondo find...,cheap flight search comp flight momondo find c...,cheap flight search compar flight momondo find...
4,bot create free account create free account si...,bot creat free account creat free account sign...,bot cre fre account cre fre account sign accou...,bot creat free account creat free account sign...
...,...,...,...,...
1403,old nude woman porn mature granny sex horny ol...,old nude women porn matur granni sex horni old...,old nud wom porn mat granny sex horny old gran...,old nude women porn matur granni sex horni old...
1404,bdsm cam bdsm chat bondage cam free bdsm video...,bdsm cam bdsm chat bondag cam free bdsm video ...,bdsm cam bdsm chat bond cam fre bdsm video cha...,bdsm cam bdsm chat bondag cam free bdsm video ...
1405,porno dvd online european porn dvd cheap adult...,porno dvd onlin european porn dvd cheap adult ...,porno dvd onlin europ porn dvd cheap adult mov...,porno dvd onlin european porn dvd cheap adult ...
1406,anal dream house anal dream house anal dream h...,anal dream hous anal dream hous anal dream hou...,an dream hous an dream hous an dream hous tant...,anal dream hous anal dream hous anal dream hou...


In [21]:
df_final = pd.concat([df2, df_stemmed], axis=1)

print("\n=== Results ===")
print(df_final[['expanded_text', 'cleaned_text', 'lemma', 'porter']].head())


=== Results ===
                                       expanded_text  \
0  official site good hotel accommodation big sav...   
1  expedia hotel book sites like use vacation wor...   
2  tripadvisor hotel book sites like previously d...   
3  cheap flights search compare flights momondo f...   
4  bot create free account create free account si...   

                                        cleaned_text  \
0  official site good hotel accommodation big sav...   
1  expedia hotel book sites like use vacation wor...   
2  tripadvisor hotel book sites like previously d...   
3  cheap flights search compare flights momondo f...   
4  bot create free account create free account si...   

                                               lemma  \
0  official site good hotel accommodation big sav...   
1  expedia hotel book site like use vacation work...   
2  tripadvisor hotel book site like previously de...   
3  cheap flight search compare flight momondo fin...   
4  bot create free account cr

In [22]:
df_final

,Unnamed: 0,website_url,cleaned_website_text,Category,Category_b,expanded_text,cleaned_text,word_tokens,sentence_tokens,correct_tokens,lemma,porter,lancaster,snowball
0,0,https://www.booking.com/index.html?aid=1743217,official site good hotel accommodation big sav...,Travel,15,official site good hotel accommodation big sav...,official site good hotel accommodation big sav...,"['official', 'site', 'good', 'hotel', 'accommo...",['official site good hotel accommodation big s...,official site good hotel accommodation big sav...,official site good hotel accommodation big sav...,offici site good hotel accommod big save hotel...,off sit good hotel accommod big sav hotel dest...,offici site good hotel accommod big save hotel...
1,1,https://travelsites.com/expedia/,expedia hotel book sites like use vacation wor...,Travel,15,expedia hotel book sites like use vacation wor...,expedia hotel book sites like use vacation wor...,"['expedia', 'hotel', 'book', 'sites', 'like', ...",['expedia hotel book sites like use vacation w...,expedia hotel book sites like use vacation wor...,expedia hotel book site like use vacation work...,expedia hotel book site like use vacat work ha...,exped hotel book sit lik us vac work hard year...,expedia hotel book site like use vacat work ha...
2,2,https://travelsites.com/tripadvisor/,tripadvisor hotel book sites like previously d...,Travel,15,tripadvisor hotel book sites like previously d...,tripadvisor hotel book sites like previously d...,"['tripadvisor', 'hotel', 'book', 'sites', 'lik...",['tripadvisor hotel book sites like previously...,tripadvisor hotel book sites like previously d...,tripadvisor hotel book site like previously de...,tripadvisor hotel book site like previous deal...,tripadv hotel book sit lik prevy deal predomin...,tripadvisor hotel book site like previous deal...
3,3,https://www.momondo.in/?ispredir=true,cheap flights search compare flights momondo f...,Travel,15,cheap flights search compare flights momondo f...,cheap flights search compare flights momondo f...,"['cheap', 'flights', 'search', 'compare', 'fli...",['cheap flights search compare flights momondo...,cheap flights search compare flights momondo f...,cheap flight search compare flight momondo fin...,cheap flight search compar flight momondo find...,cheap flight search comp flight momondo find c...,cheap flight search compar flight momondo find...
4,4,https://www.ebookers.com/?AFFCID=EBOOKERS-UK.n...,bot create free account create free account si...,Travel,15,bot create free account create free account si...,bot create free account create free account si...,"['bot', 'create', 'free', 'account', 'create',...",['bot create free account create free account ...,bot create free account create free account si...,bot create free account create free account si...,bot creat free account creat free account sign...,bot cre fre account cre fre account sign accou...,bot creat free account creat free account sign...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1403,1403,http://www.oldwomen.org/,old nude women porn mature granny sex horny ol...,Adult,0,old nude women porn mature granny sex horny ol...,old nude women porn mature granny sex horny ol...,"['old', 'nude', 'women', 'porn', 'mature', 'gr...",['old nude women porn mature granny sex horny ...,old nude women porn mature granny sex horny ol...,old nude woman porn mature granny sex horny ol...,old nude women porn matur granni sex horni old...,old nud wom porn mat granny sex horny old gran...,old nude women porn matur granni sex horni old...
1404,1404,http://www.webcamslave.com,bdsm cams bdsm chat bondage cams free bdsm vid...,Adult,0,bdsm cams bdsm chat bondage cams free bdsm vid...,bdsm cams bdsm chat bondage cams free bdsm vid...,"['bdsm', 'cams', 'bdsm', 'chat', 'bondage', 'c...",['bdsm cams bdsm chat bondage cams free bdsm v...,bdsm cams bdsm chat bondage cams free bdsm vid...,bdsm cam bdsm chat bondage cam free bdsm video...,bdsm cam bdsm chat bondag cam free bdsm video 

In [23]:
#create vocab
from collections import Counter
import nltk

def create_vocabulary(texts, min_freq=2, max_vocab_size=10000):
    all_tokens = []
    for text in texts:
        tokens = nltk.word_tokenize(text.lower())
        all_tokens.extend(tokens)

    token_counts = Counter(all_tokens)
    # Fix the typo here
    vocab = {token: count for token, count in token_counts.most_common(max_vocab_size) if count >= min_freq}

    token2id = {token: idx for idx, (token, _) in enumerate(vocab.items())}
    id2token = {idx: token for token, idx in token2id.items()}

    return vocab, token2id, id2token

vocab, token2id, id2token = create_vocabulary(df_final['lemma'], min_freq=2)

In [25]:
# See what categories exist
print(df['Category'].value_counts())
print(f"\nTotal websites: {len(df)}")
print(f"Total categories: {df['Category'].nunique()}")

Category
Education                          114
Business/Corporate                 109
Travel                             107
Streaming Services                 105
Sports                             104
E-Commerce                         102
Games                               98
News                                96
Health and Fitness                  96
Photography                         93
Computers and Technology            93
Food                                92
Law and Government                  84
Social Networking and Messaging     83
Forums                              16
Adult                               16
Name: count, dtype: int64

Total websites: 1408
Total categories: 16


In [26]:
from collections import Counter
import nltk

def create_category_vocabularies(df, text_col='lemma', category_col='Category', min_freq=2):
    category_vocab = {}

    for category in df[category_col].unique():
        texts = df[df[category_col] == category][text_col]

        all_tokens = []
        for text in texts:
            tokens = nltk.word_tokenize(str(text).lower())
            all_tokens.extend(tokens)

        token_counts = Counter(all_tokens)

        vocab = {token: count for token, count in token_counts.items() if count >= min_freq}

        category_vocab[category] = vocab

    return category_vocab


category_vocabs = create_category_vocabularies(df_final)

for cat, vocab in category_vocabs.items():
    top_words = sorted(vocab.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\n{cat}:")
    print(top_words)


Travel:
[('hotel', 1146), ('travel', 829), ('flight', 808), ('tour', 789), ('resort', 559), ('book', 529), ('room', 511), ('view', 463), ('day', 438), ('new', 433)]

Social Networking and Messaging:
[('chat', 1587), ('room', 721), ('free', 365), ('online', 278), ('new', 248), ('people', 217), ('service', 178), ('friend', 170), ('member', 155), ('site', 154)]

News:
[('news', 2941), ('new', 1091), ('world', 1036), ('late', 827), ('coronavirus', 657), ('vaccine', 625), ('trump', 611), ('december', 608), ('business', 590), ('ago', 585)]

Streaming Services:
[('tv', 1182), ('movie', 938), ('live', 733), ('video', 642), ('watch', 619), ('series', 517), ('free', 479), ('music', 410), ('stream', 381), ('comedy', 361)]

Sports:
[('sport', 1339), ('league', 1274), ('december', 1170), ('cricket', 956), ('news', 897), ('football', 820), ('team', 793), ('v', 694), ('match', 626), ('player', 554)]

Photography:
[('photography', 1174), ('photographer', 766), ('photo', 750), ('camera', 650), ('image

In [49]:
import pandas as pd

# Dictionary to store a DataFrame for each category
category_vocab_dfs = {}

for category, vocab in category_vocabs.items():
    # Convert vocab dictionary to a DataFrame
    df = pd.DataFrame(list(vocab.items()), columns=['Word', 'Count'])
    # Sort by count descending
    df = df.sort_values(by='Count', ascending=False).reset_index(drop=True)
    # Store in dictionary
    category_vocab_dfs[category] = df

# Example: view Travel category DataFrame
print("Travel category vocabulary:")
print(category_vocab_dfs['Travel'].head(10))

# Example: view Social Networking and Messaging
print("\nSocial Networking and Messaging category vocabulary:")
print(category_vocab_dfs['Social Networking and Messaging'].head(10))

Travel category vocabulary:
     Word  Count
0   hotel   1146
1  travel    829
2  flight    808
3    tour    789
4  resort    559
5    book    529
6    room    511
7    view    463
8     day    438
9     new    433

Social Networking and Messaging category vocabulary:
      Word  Count
0     chat   1587
1     room    721
2     free    365
3   online    278
4      new    248
5   people    217
6  service    178
7   friend    170
8   member    155
9      use    154


In [52]:
education_df = category_vocab_dfs['Education']
sports_df = category_vocab_dfs['Sports']
travel_df = category_vocab_dfs['Travel']
services_df = category_vocab_dfs['Streaming Services']
ecommerce_df = category_vocab_dfs['E-Commerce']
games_df = category_vocab_dfs['Games']
news_df = category_vocab_dfs['News']
fitness_df = category_vocab_dfs['Health and Fitness']
photography_df = category_vocab_dfs['Photography']
computers_df = category_vocab_dfs['Computers and Technology']
food_df = category_vocab_dfs['Food']
law_df = category_vocab_dfs['Law and Government']
social_networking_df = category_vocab_dfs['Social Networking and Messaging']
forum_df = category_vocab_dfs['Forums']
adult_df = category_vocab_dfs['Adult']

In [53]:
education_df

,Word,Count
0,edit,509
1,science,459
2,student,452
3,group,452
4,priestley,431
...,...,...
7240,certified,2
7241,viable,2
7242,dupont,2
7243,teflon,2


In [54]:
def predict_category(text, category_vocabs):
    tokens = nltk.word_tokenize(text.lower())

    scores = {}

    for category, vocab in category_vocabs.items():
        score = sum(1 for token in tokens if token in vocab)
        scores[category] = score

    return max(scores, key=scores.get), scores

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

X = df_final['lemma']      # text
y = df_final['Category']   # labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=10000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LinearSVC()

model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9432624113475178
                                 precision    recall  f1-score   support

                          Adult       1.00      0.67      0.80         3
             Business/Corporate       0.91      0.91      0.91        22
       Computers and Technology       0.89      0.89      0.89        19
                     E-Commerce       0.95      0.95      0.95        20
                      Education       0.96      0.96      0.96        23
                           Food       0.90      1.00      0.95        18
                         Forums       1.00      0.67      0.80         3
                          Games       1.00      0.95      0.97        20
             Health and Fitness       0.94      0.89      0.92        19
             Law and Government       0.94      1.00      0.97        17
                           News       0.95      0.95      0.95        19
                    Photography       1.00      0.89      0.94        19
Social Networking and

In [56]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train_vec, y_train, cv=50)
print(scores.mean())

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 13 members, which is less than n_splits=50.
  warnings.warn(


0.9302766798418971


In [57]:
def predict_category(text):
    text_vec = vectorizer.transform([text])
    return model.predict(text_vec)[0]

predict_category("watch live football match and league results")

'Sports'

In [58]:
df_final['Predicted_Category'] = model.predict(vectorizer.transform(df_final['lemma']))
df_final['Correct'] = df_final['Category'] == df_final['Predicted_Category']
df_final['Correct'].value_counts()
accuracy = df_final['Correct'].mean()
print("Accuracy on all the dataset:", accuracy)
errors = df_final[df_final['Correct'] == False]
errors['Category'].value_counts()

Accuracy on all the dataset: 0.9879261363636364


Category
Social Networking and Messaging    2
Photography                        2
Health and Fitness                 2
Computers and Technology           2
Business/Corporate                 2
Streaming Services                 1
News                               1
Games                              1
Forums                             1
Education                          1
E-Commerce                         1
Adult                              1
Name: count, dtype: int64

In [59]:
def predict_website_category_with_confidence(text, model=model, vectorizer=vectorizer):
    text_vec = vectorizer.transform([text])
    predicted_category = model.predict(text_vec)[0]
    
    # Get decision scores for all categories
    scores = model.decision_function(text_vec)
    # Highest score is the confidence
    confidence = scores.max()
    
    return predicted_category, confidence

category, confidence = predict_website_category_with_confidence("new smartphone reviews and tech news")
print(category)
print(confidence)

News
-0.09230031105116132


In [61]:
# Function to predict category
def predict_website_category(text, model=model, vectorizer=vectorizer):
    text_vec = vectorizer.transform([text])
    return model.predict(text_vec)[0]

# Let the user enter text
text_example = input("Enter website text: ")
category = predict_website_category(text_example)
print("Predicted category:", category)

Enter website text:  new smartphone reviews and tech news


Predicted category: News
